# 🚀 Notebook 03: Supervised Classification Pipeline
## *PRML Benchmark: 12 Supervised Classifiers with Stratified 5-Fold Cross-Validation*

---
### 🎯 Endsem & PRML Algorithms Covered:
- **Probabilistic Models**: Gaussian Naive Bayes
- **Linear Models**: Logistic Regression
- **Kernel Methods**: SVM with Radial Basis Function (RBF) & Polynomial Kernels
- **Decision Trees**: Gini Impurity, depth-regulated non-parametric models
- **Instance-Based**: KNN ($k=1, 3, 5$) and Distance-Weighted KNN
- **Ensemble Methods**: Random Forest & AdaBoost (SAMME.R)

In [ ]:
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


warnings.filterwarnings('ignore')
os.makedirs('../models', exist_ok=True)
os.makedirs('../presentation', exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

# Load clean partitions
X_train, y_train, X_test, y_test = joblib.load('../data/processed_data.pkl')

# Label encoding for multiclass targets
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Target Classes Mapped: {dict(enumerate(le.classes_))}")
print(f"Training Matrix: {X_train_scaled.shape}, Test Matrix: {X_test_scaled.shape}")

## 1️⃣ Model Definitions & Stratified 5-Fold Cross-Validation

In [ ]:
models = {
    'Gaussian Naive Bayes': GaussianNB(),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'KNN (k=1)': KNeighborsClassifier(n_neighbors=1),
    'KNN (k=3)': KNeighborsClassifier(n_neighbors=3),
    'KNN (k=5)': KNeighborsClassifier(n_neighbors=5),
    'Weighted KNN (k=3)': KNeighborsClassifier(n_neighbors=3, weights='distance'),
    'Weighted KNN (k=5)': KNeighborsClassifier(n_neighbors=5, weights='distance'),
    'SVM (RBF Kernel)': SVC(kernel='rbf', C=10.0, random_state=42),
    'SVM (Poly Kernel)': SVC(kernel='poly', degree=3, C=1.0, random_state=42),
    'Random Forest (100 Trees)': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'AdaBoost Classifier': AdaBoostClassifier(n_estimators=100, random_state=42)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

print("=" * 65)
print("STRATIFIED 5-FOLD CROSS-VALIDATION ACROSS ALL 561 FEATURES")
print("=" * 65)

for name, clf in models.items():
    scores = cross_val_score(make_pipeline(StandardScaler(), clf), X_train, y_train_enc, cv=cv, scoring='accuracy', n_jobs=-1)
    cv_results[name] = {
        'Mean Accuracy': scores.mean(),
        'Std Dev': scores.std()
    }
    print(f"{name:28s} : {scores.mean()*100:.2f}% ± {scores.std()*100:.2f}%")

df_results = pd.DataFrame(cv_results).T.sort_values(by='Mean Accuracy', ascending=False)
print("\nRanked Performance:")
print(df_results)

## 2️⃣ Performance Visualization: Model Comparison

### Recorded results — static preview
![Recorded baseline comparison](../presentation/09_recorded_baseline_cv.svg)

Selected historical baseline scores from the project presentation and the updated Logistic Regression benchmark. These are not a fresh execution of all 12 models; preprocessing protocols differ. Run the notebook to regenerate the full comparison.


In [ ]:
plt.figure(figsize=(12, 6))
bars = plt.barh(df_results.index[::-1], df_results['Mean Accuracy'][::-1], 
                xerr=df_results['Std Dev'][::-1], color='#2b5c8f', edgecolor='black', alpha=0.85, capsize=4)
plt.xlabel('Stratified 5-Fold Cross-Validation Accuracy', fontsize=11, fontweight='bold')
plt.title('Baseline Algorithm Performance Comparison (All 561 Features)', fontsize=13, fontweight='bold')
plt.xlim(0, 1.08)
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w*100:.2f}%", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('../presentation/09_baseline_cv_comparison.png', dpi=200)
plt.show()

## 3️⃣ Test Set Generalization & Best Model Confusion Matrix

In [ ]:
# Fit top performers on full training set and evaluate on unseen test set (2,947 samples)
top_model = SVC(kernel='rbf', C=10.0, probability=True, random_state=42)
top_model.fit(X_train_scaled, y_train_enc)
y_pred = top_model.predict(X_test_scaled)

acc = accuracy_score(y_test_enc, y_pred)
f1 = f1_score(y_test_enc, y_pred, average='macro')
prec = precision_score(y_test_enc, y_pred, average='macro')
rec = recall_score(y_test_enc, y_pred, average='macro')

print("=" * 50)
print("🏆 SVM (RBF) TEST SET EVALUATION")
print("=" * 50)
print(f"Test Accuracy : {acc*100:.2f}%")
print(f"Macro F1-Score: {f1*100:.2f}%")
print(f"Precision     : {prec*100:.2f}%")
print(f"Recall        : {rec*100:.2f}%")

# High-definition confusion matrix
cm = confusion_matrix(y_test_enc, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=le.classes_, yticklabels=le.classes_)
plt.title(f'Confusion Matrix — SVM (RBF Kernel) | Test Accuracy = {acc*100:.2f}%', fontweight='bold')
plt.xlabel('Predicted Activity Class', fontweight='bold')
plt.ylabel('Ground-Truth Activity Class', fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../presentation/10_best_model_confusion_matrix.png', dpi=200)
plt.show()

# Serialize model artifacts
joblib.dump({
    'cv_results': cv_results,
    'best_model': top_model,
    'scaler': scaler,
    'label_encoder': le
}, '../models/modeling_results.pkl')
print('✅ Modeling artifacts saved to ../models/modeling_results.pkl')